# Data Preparation

This notebook reads the extracted RGB images and masks, validates image-mask pairs, creates the fixed source-image split, and builds finite TensorFlow datasets for the later baseline, CNN, and U-Net notebooks.

In [ ]:
from pathlib import Path
import json
import math
import numpy as np
from PIL import Image
import tensorflow as tf

SEED = 42
PATCH_SIZE = 256
BATCH_SIZE = 8
IGNORE_LABEL = 255
CLASS_COLORS = np.array([[199, 199, 199], [31, 119, 180], [255, 127, 14]], dtype=np.uint8)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'project_plan.md').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'final-project'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
ARTIFACT_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
if not RAW_DIR.exists():
    raise FileNotFoundError(f'Raw data directory not found: {RAW_DIR}')
print('Project root:', PROJECT_ROOT)
print('Raw data:', RAW_DIR)


## Image-mask validation

In [ ]:
def paired_files(group):
    folder = RAW_DIR / group
    images = {p.stem: p for p in (folder / 'img').glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}}
    masks = {p.stem: p for p in (folder / 'msk').glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}}
    if set(images) != set(masks):
        raise ValueError(f'{group}: image-mask identifiers do not match')
    return sorted((images[key], masks[key]) for key in images)

groups = ['trainval', 'test']
validation_rows = []
for group in groups:
    for image_path, mask_path in paired_files(group):
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            image_rgb = np.asarray(image.convert('RGB'))
            mask_rgb = np.asarray(mask.convert('RGB'))
        if image_rgb.shape != mask_rgb.shape or image_rgb.shape[2] != 3:
            raise ValueError(f'Shape mismatch: {image_path}')
        valid = np.zeros(mask_rgb.shape[:2], dtype=bool)
        for color in CLASS_COLORS:
            valid |= np.all(mask_rgb == color, axis=-1)
        if not valid.all():
            raise ValueError(f'Unexpected mask colors: {mask_path}')
        validation_rows.append({'group': group, 'source_id': image_path.stem, 'width': image_rgb.shape[1], 'height': image_rgb.shape[0]})
print(f'Validated {len(validation_rows)} image-mask pairs.')
print({group: sum(row['group'] == group for row in validation_rows) for group in groups})
(ARTIFACT_DIR / 'DATA_VALIDATION.json').write_text(json.dumps(validation_rows, indent=2) + '\n', encoding='utf-8')


## Fixed source-image split

In [ ]:
trainval_images = sorted((RAW_DIR / 'trainval' / 'img').glob('*.jpg'))
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(trainval_images))
val_images = [trainval_images[i] for i in indices[:3]]
train_images = [trainval_images[i] for i in indices[3:]]
def mask_for(image_path):
    return image_path.parent.parent / 'msk' / f'{image_path.stem}.png'
train_masks = [mask_for(path) for path in train_images]
val_masks = [mask_for(path) for path in val_images]
split_record = {'seed': SEED, 'train_images': [p.relative_to(PROJECT_ROOT).as_posix() for p in train_images], 'validation_images': [p.relative_to(PROJECT_ROOT).as_posix() for p in val_images]}
(ARTIFACT_DIR / 'split.json').write_text(json.dumps(split_record, indent=2) + '\n', encoding='utf-8')
print('Training source images:', len(train_images))
print('Validation source images:', len(val_images))


## Patch generation and TensorFlow datasets

In [ ]:
def generate_patches(image_paths, mask_paths):
    for image_path, mask_path in zip(image_paths, mask_paths):
        with Image.open(image_path) as image, Image.open(mask_path) as mask:
            rgb = np.asarray(image.convert('RGB'))
            mask_rgb = np.asarray(mask.convert('RGB'))
        labels = np.full(mask_rgb.shape[:2], IGNORE_LABEL, dtype=np.uint8)
        for class_id, color in enumerate(CLASS_COLORS):
            labels[np.all(mask_rgb == color, axis=-1)] = class_id
        for y in range(0, labels.shape[0], PATCH_SIZE):
            for x in range(0, labels.shape[1], PATCH_SIZE):
                height = min(PATCH_SIZE, labels.shape[0] - y)
                width = min(PATCH_SIZE, labels.shape[1] - x)
                image_patch = np.zeros((PATCH_SIZE, PATCH_SIZE, 3), dtype=np.float32)
                label_patch = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.int32)
                weight_patch = np.zeros((PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
                image_patch[:height, :width] = rgb[y:y + height, x:x + width] / 255.0
                label_patch[:height, :width] = labels[y:y + height, x:x + width]
                weight_patch[:height, :width] = 1.0
                yield image_patch, label_patch, weight_patch

def count_patches(paths):
    total = 0
    for path in paths:
        with Image.open(path) as image:
            total += math.ceil(image.height / PATCH_SIZE) * math.ceil(image.width / PATCH_SIZE)
    return total

def make_dataset(image_paths, mask_paths, training=False):
    dataset = tf.data.Dataset.from_generator(lambda: generate_patches(image_paths, mask_paths), output_signature=(tf.TensorSpec((PATCH_SIZE, PATCH_SIZE, 3), tf.float32), tf.TensorSpec((PATCH_SIZE, PATCH_SIZE), tf.int32), tf.TensorSpec((PATCH_SIZE, PATCH_SIZE), tf.float32)))
    if training:
        dataset = dataset.shuffle(64, seed=SEED, reshuffle_each_iteration=True)
    return dataset.batch(BATCH_SIZE)

train_dataset = make_dataset(train_images, train_masks, training=True)
val_dataset = make_dataset(val_images, val_masks)
train_steps = math.ceil(count_patches(train_images) / BATCH_SIZE)
val_steps = math.ceil(count_patches(val_images) / BATCH_SIZE)
print('Training steps:', train_steps)
print('Validation steps:', val_steps)
batch_images, batch_labels, batch_weights = next(iter(train_dataset))
print('Batch shapes:', batch_images.shape, batch_labels.shape, batch_weights.shape)
